# Notebook for causal analysis for the AGE subgroups using DoWhy-library

In [1]:
from dowhy import CausalModel
import pandas as pd

### Define outcome and the confounders for each feature

In [2]:
outcome = "How happy are you?"

In [3]:
feature_confounder_map = {
    "Health condition": [
        "Age",
        "Income quartiles",
        "Chronic health problems?",
        "Education completed",
        "Employment - 7 groups"
    ],

    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "How frequently participate in social activities?",
        "Employment - 7 groups",
        "Marital status",
        "Health condition"
    ],

    "Can't find the way because life has become so complicated?": [
        "Education completed",
        "Employment - 7 groups",
        "A person to get support from when feeling depressed",
        "Age",
        "Household size"
    ],

    "I feel I am free to decide how to live my life": [
        "Income quartiles",
        "Education completed",
        "How much trust the government?"
    ],

    "I am optimistic about the future": [
        "Personal financial situation",
        "Access to recreational or green areas?",
        "A person to get support from to raise emergency money",
        "Health condition",
        "Age",
        "Employment - 7 groups"
    ],

    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Personal financial situation",
        "Household size",
        "No. of children"
    ],

    "Personal financial situation": [
        "Employment - 7 groups",
        "Can afford a meal with meat/chicken/fish every second day?",
        "Household structure",
        "Income quartiles",
        "Education completed"
    ],

    "How much trust the police?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the government?",
        "How much trust the legal system?",
        "Rural/urban living",
        "Age"
    ],

    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?",
        "Household size",
        "Household structure"
    ],

    "Quality of education system?": [
        "Education completed",
        "How much trust the legal system?",
        "Income quartiles",
        "Rural/urban living",
        "Age"
    ],

    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed",
        "Education completed",
        "Marital status"
    ],

    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?",
        "No. of children",
        "Education completed"
    ],

    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?",
        "Education completed",
        "Rural/urban living"
    ]
}

### Load data, recode variables, and define helper function

In [4]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]

# Binary marital status: 1 = married/living with partner, 0 = all other categories
data["Marital status"] = (data["Marital status"] == 1).astype(int)

# Age subgroup variable
age_column = "Age"

# Important: Age is the subgroup variable here.
# Therefore, Age is removed from the confounder sets within the age-specific models,
# because it is constant inside each subgroup.
def clean_confounders_for_age_subgroups(treatment, confounders):
    return [c for c in confounders if c not in [age_column, treatment]]


def calculate_causal_values(data_source, confounder_map):
    # Save results
    results = []

    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        adjusted_confounders = clean_confounders_for_age_subgroups(treatment, confounders)

        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data_source,
                treatment=treatment,
                outcome=outcome,
                common_causes=adjusted_confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders": ", ".join(adjusted_confounders)
            })

        except Exception as e:
            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders": ", ".join(adjusted_confounders),
                "Error": str(e)
            })

    return results

### Create age subgroups

In [5]:
# Age subgroups
df_age1 = data[data[age_column] == 1].copy()
df_age2 = data[data[age_column] == 2].copy()
df_age3 = data[data[age_column] == 3].copy()
df_age4 = data[data[age_column] == 4].copy()
df_age5 = data[data[age_column] == 5].copy()

age_subgroups = {
    "age1": df_age1,
    "age2": df_age2,
    "age3": df_age3,
    "age4": df_age4,
    "age5": df_age5,
}

for group_name, group_df in age_subgroups.items():
    print(group_name, group_df.shape)

age1 (520, 168)
age2 (1099, 168)
age3 (2066, 168)
age4 (1767, 168)
age5 (670, 168)


### Calculate ATEs for each age subgroup

In [6]:
age_results = {}

for group_name, group_df in age_subgroups.items():
    print(f"\n===== {group_name} =====")
    df_results = pd.DataFrame(calculate_causal_values(group_df, feature_confounder_map))
    df_results.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
    age_results[group_name] = df_results
    display(df_results)


===== age1 =====
Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders
0,Health condition,0.584877,"Income quartiles, Chronic health problems?, Ed..."
1,I generally feel that what I do in life is wor...,0.579027,A person to get support from when feeling depr...
3,I feel I am free to decide how to live my life,0.478474,"Income quartiles, Education completed, How muc..."
4,I am optimistic about the future,0.467019,"Personal financial situation, Access to recrea..."
6,Personal financial situation,0.358924,"Employment - 7 groups, Can afford a meal with ..."
10,The value of what I do is not recognised by ot...,0.351814,"Employment - 7 groups, How frequently particip..."
8,Deprivation index: No. of items hhold can't af...,-0.319609,"Income quartiles, Employment - 7 groups, Can a..."
5,Household able to make ends meet?,0.299871,"Employment - 7 groups, Income quartiles, Perso..."
2,Can't find the way because life has become so ...,-0.266097,"Education completed, Employment - 7 groups, A ..."
11,Marital status,0.221336,"Household structure, How frequently participat..."


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers


===== age2 =====
Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


...

,Treatment,ATE (DML),Confounders
11,Marital status,0.818013,"Household structure, How frequently participat..."
0,Health condition,0.654103,"Income quartiles, Chronic health problems?, Ed..."
1,I generally feel that what I do in life is wor...,0.634844,A person to get support from when feeling depr...
3,I feel I am free to decide how to live my life,0.543465,"Income quartiles, Education completed, How muc..."
6,Personal financial situation,0.481127,"Employment - 7 groups, Can afford a meal with ..."
2,Can't find the way because life has become so ...,-0.450264,"Education completed, Employment - 7 groups, A ..."
4,I am optimistic about the future,0.434658,"Personal financial situation, Access to recrea..."
10,The value of what I do is not recognised by ot...,0.381734,"Employment - 7 groups, How frequently particip..."
5,Household able to make ends meet?,0.357067,"Employment - 7 groups, Income quartiles, Perso..."
8,Deprivation index: No. of items hhold can't af...,-0.354229,"Income quartiles, Employment - 7 groups, Can a..."


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers


===== age5 =====
Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders
0,Health condition,0.668746,"Income quartiles, Chronic health problems?, Ed..."
6,Personal financial situation,0.559952,"Employment - 7 groups, Can afford a meal with ..."
1,I generally feel that what I do in life is wor...,0.534924,A person to get support from when feeling depr...
2,Can't find the way because life has become so ...,-0.510906,"Education completed, Employment - 7 groups, A ..."
3,I feel I am free to decide how to live my life,0.475449,"Income quartiles, Education completed, How muc..."
8,Deprivation index: No. of items hhold can't af...,-0.402749,"Income quartiles, Employment - 7 groups, Can a..."
10,The value of what I do is not recognised by ot...,0.378445,"Employment - 7 groups, How frequently particip..."
5,Household able to make ends meet?,0.309425,"Employment - 7 groups, Income quartiles, Perso..."
4,I am optimistic about the future,0.271357,"Personal financial situation, Access to recrea..."
9,Quality of education system?,0.248711,"Education completed, How much trust the legal ..."


### Combined Results

In [7]:
# Create one comparison table with one ATE column per age subgroup
comparison_tables = []

for group_name, df_results in age_results.items():
    temp = df_results[["Treatment", "ATE (DML)"]].rename(
        columns={"ATE (DML)": f"ATE_{group_name}"}
    )
    comparison_tables.append(temp)

from functools import reduce

df_compare_age = reduce(
    lambda left, right: pd.merge(left, right, on="Treatment", how="outer"),
    comparison_tables
)

# Sort by absolute ATE in age1 as a simple default
# can change this to another subgroup or to the mean absolute ATE.
ate_columns = [col for col in df_compare_age.columns if col.startswith("ATE_")]
df_compare_age["Mean_abs_ATE"] = df_compare_age[ate_columns].abs().mean(axis=1)
df_compare_age = df_compare_age.sort_values("Mean_abs_ATE", ascending=False)

display(df_compare_age)

,Treatment,ATE_age1,ATE_age2,ATE_age3,ATE_age4,ATE_age5,Mean_abs_ATE
3,Health condition,0.584877,0.479999,0.745427,0.654103,0.668746,0.626630
8,I generally feel that what I do in life is wor...,0.579027,0.566913,0.613360,0.634844,0.534924,0.585814
9,Marital status,0.221336,0.526652,0.915750,0.818013,0.186856,0.533721
7,I feel I am free to decide how to live my life,0.478474,0.460224,0.490641,0.543465,0.475449,0.489650
10,Personal financial situation,0.358924,0.309194,0.580974,0.481127,0.559952,0.458034
1,Can't find the way because life has become so ...,-0.266097,-0.329612,-0.505909,-0.450264,-0.510906,0.412558
6,I am optimistic about the future,0.467019,0.380380,0.371945,0.434658,0.271357,0.385072
12,The value of what I do is not recognised by ot...,0.351814,0.350376,0.445360,0.381734,0.378445,0.381546
4,Household able to make ends meet?,0.299871,0.306372,0.347141,0.357067,0.309425,0.323975
2,Deprivation index: No. of items hhold can't af...,-0.319609,-0.183011,-0.319746,-0.354229,-0.402749,0.315869


### Export Results

In [8]:
# Export combined results
df_compare_age.to_csv("Results/results_age_causal_analysis.csv", index=False)

# Optional: export each subgroup table separately
#for group_name, df_results in age_results.items():
#    df_results.to_csv(f"Results/results_{group_name}_causal_analysis.csv", index=False)